# 02 — Advanced Classification

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Train the classifier, inspect its confidence and uncertainty guard, and compare four models.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### Train and predict
The classifier reports the category, a confidence, the runner-up, and how many words it recognized.

In [2]:
from src.analysis.classifier import NewsClassifier
clf = NewsClassifier().fit(df["text"], df["category"])
for t in ["The Nasdaq fell as Apple shares dropped on cost worries.",
          "The striker scored twice in the cup final."]:
    print(t)
    print("  ->", clf.predict(t), "\n")

The Nasdaq fell as Apple shares dropped on cost worries.
  -> {'category': 'business', 'confidence': 0.615, 'runner_up': 'entertainment (0.11)', 'recognized_terms': 7, 'note': '', 'key_terms': ['nasdaq', 'apple', 'worry', 'dropped', 'fell', 'share']} 

The striker scored twice in the cup final.
  -> {'category': 'sport', 'confidence': 0.931, 'runner_up': 'entertainment (0.02)', 'recognized_terms': 6, 'note': '', 'key_terms': ['cup final', 'striker', 'scored', 'twice', 'cup', 'final']} 



### Uncertainty guard
On empty or out-of-scope input the classifier returns `uncertain` instead of guessing. This is the fix for the midterm bug where short text was labeled sport.

In [3]:
for t in ["", "aaa bbb ccc", "the the the"]:
    print(repr(t), "->", clf.predict(t)["category"])

'' -> uncertain
'aaa bbb ccc' -> uncertain
'the the the' -> uncertain


### Model comparison
Four classifiers, cross-validated. Linear SVM and Logistic Regression lead.

In [4]:
cmp = clf.compare_models(df["text"], df["category"], cv=5)
import pandas as pd
pd.DataFrame(cmp)

,model,cv_accuracy,test_accuracy,macro_f1
0,Logistic Regression,0.980,0.991,0.990
1,Linear SVM,0.981,0.987,0.986
2,Multinomial NB,0.969,0.982,0.982
3,Random Forest,0.958,0.973,0.973


**Takeaway.** TF-IDF plus a linear model classifies BBC news at high accuracy, and the uncertainty guard keeps it honest on inputs it was not trained for.